# 03 - LoRA fine-tuning of Qwen2.5-0.5B-Instruct (local)

**Goal:** teach a small LLM to turn OCR-like invoice text into the strict JSON schema of this project.

**Runs locally** from the `notebooks/` folder of the project. It auto-detects the device: NVIDIA GPU (`cuda`), Apple Silicon (`mps`) or `cpu`.

Before starting:
- `pip install jupyter torch transformers peft accelerate` (or open the notebook in VS Code with a Python kernel).
- Internet is needed once to download the base model (about 1 GB) from Hugging Face.
- On a Mac, expect **~30-60 min** with the default `TRAIN_SIZE`. Watch the progress bar ETA: if it is too slow, interrupt, lower `TRAIN_SIZE`/`EPOCHS` in the configuration cell and re-run from there. On CPU only, use `TRAIN_SIZE = 300` for a first test.

In [ ]:
# Must run before importing torch: let PyTorch fall back to CPU for ops not yet supported on Apple's MPS.
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Uncomment on the first run if the libraries are missing:
# %pip install -q torch transformers peft accelerate

In [ ]:
import sys
from pathlib import Path

# The notebook lives in <project>/notebooks/, so the project root is the parent folder.
REPO = Path.cwd().parent if (Path.cwd().parent / "src").exists() else Path.cwd()
assert (REPO / "src" / "generator.py").exists(), f"Project root not found from {Path.cwd()}"
sys.path.insert(0, str(REPO))
print("Project root:", REPO)

In [ ]:
import random
import numpy as np
import torch
import transformers, peft

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("Device:", DEVICE)

# ---------------- Configuration ----------------
MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
SEED         = 42
N_RECORDS    = 5000        # same as scripts/generate_dataset.py default -> identical splits to data/*.json
TRAIN_SIZE   = 200        # how many of the 4000 training invoices to use (raise it if your machine is fast)
MAX_LEN      = 1536        # longer examples are dropped (never truncated: that would cut the JSON)
EPOCHS       = 1
LR           = 2e-4
BATCH        = 2
GRAD_ACC     = 2           # effective batch size = BATCH * GRAD_ACC = 16
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
OUT_DIR      = REPO / "models" / "qwen-invoice-lora"   # where the app expects the adapter

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Data
We regenerate the synthetic dataset with the same seed as `scripts/generate_dataset.py`, so train/validation/test are identical to `data/*.json`. The test split is never used for training.

In [ ]:
from src.generator import generate_dataset
from src.preprocessing import split_dataset, to_messages, target_json

records = generate_dataset(N_RECORDS, seed=SEED)
train, val, test = split_dataset(records, seed=SEED)
train = train[:TRAIN_SIZE]
print(f"train={len(train)} validation={len(val)} test={len(test)}")
print(train[0]["input"][:400])

## 2. Tokenization with loss masking
The loss is computed **only on the assistant answer (the JSON)**. Prompt tokens get label `-100`, so the model learns to produce the JSON, not to repeat the invoice.

The prompt is built with `to_messages()` from `src/preprocessing.py`, the exact same function inference uses.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print("eos:", tok.eos_token, "| pad:", tok.pad_token)

In [ ]:
def encode(record):
    prompt_text = tok.apply_chat_template(
        to_messages(record, include_answer=False), tokenize=False, add_generation_prompt=True
    )
    answer_text = target_json(record["output"]) + tok.eos_token  # eos teaches the model to stop
    prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tok(answer_text, add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + answer_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

def build_dataset(recs, name):
    encoded = [encode(r) for r in recs]
    lengths = np.array([len(e["input_ids"]) for e in encoded])
    kept = [e for e in encoded if len(e["input_ids"]) <= MAX_LEN]
    print(f"{name}: {len(kept)}/{len(encoded)} kept | tokens mean={lengths.mean():.0f} "
          f"p95={np.percentile(lengths, 95):.0f} max={lengths.max()}")
    return kept

train_ds = build_dataset(train, "train")
val_ds = build_dataset(val[:100], "val")  # small subset keeps evaluation fast

# Sanity check: decode only the supervised part, it must be the target JSON.
ex = train_ds[0]
supervised = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print(tok.decode(supervised))

## 3. Model + LoRA
The base model is trained in float32 (the safe choice on Apple MPS and CPU; on NVIDIA GPUs mixed precision is enabled automatically). Only the small LoRA adapter weights are trained.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 4. Training
Watch the progress bar ETA. The training loss should drop clearly during the first ~50-100 optimizer steps.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

on_cuda = DEVICE == "cuda"
use_bf16 = on_cuda and torch.cuda.is_bf16_supported()
use_fp16 = on_cuda and not use_bf16
args = TrainingArguments(
    output_dir=str(REPO / "models" / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="no",   # needs transformers >= 4.41
    eval_steps=50,
    save_strategy="no",      # we save only the final adapter
    fp16=use_fp16,
    bf16=use_bf16,
    dataloader_pin_memory=False,
    report_to="none",
    seed=SEED,
)
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator)
import time
from transformers import TrainerCallback

class PrintCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()
        print(f"Début de l'entraînement : {state.max_steps} étapes au total", flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        print(f"étape {state.global_step}/{state.max_steps} | {time.time() - self.t0:.0f}s écoulées", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        print("   ->", logs, flush=True)

trainer.add_callback(PrintCallback())
trainer.train()

In [ ]:
import time
batch = collator([train_ds[i] for i in range(BATCH)])
batch = {k: v.to(DEVICE) for k, v in batch.items()}
print("forme:", batch["input_ids"].shape, "| device:", DEVICE, "| modèle sur:", next(model.parameters()).device)
model.train()
for i in range(3):
    t = time.time()
    out = model(**batch)
    out.loss.backward()
    if DEVICE == "mps":
        torch.mps.synchronize()
    print(f"essai {i}: loss={out.loss.item():.3f} | {time.time()-t:.1f}s")
    model.zero_grad()

## 5. Save the adapter

In [ ]:
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("Adapter saved to", OUT_DIR)

## 6. Quick sanity check
A first look at quality on 10 test invoices (generation is slow on a laptop, so we keep it small). The full base-vs-fine-tuned comparison belongs in `04_evaluation.ipynb`.

In [ ]:
import gc
from src.inference import InvoiceExtractor
from src.evaluation import evaluate
from src.validation import validate_invoice

del model, trainer
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

extractor = InvoiceExtractor(base_model=MODEL_NAME, adapter_path=str(OUT_DIR), device=DEVICE)
sample = test[:10]
raw_outputs = [extractor.extract(r["input"])[0] for r in sample]
scores = evaluate(raw_outputs, [r["output"] for r in sample])
print({k: round(v, 3) for k, v in scores.items()})

print("\n--- Example ---")
print(sample[0]["input"])
print(raw_outputs[0])
print("Validation issues:", validate_invoice(raw_outputs[0]).issues)